In [0]:
df = spark.table("silver_transactions")
display(df)

In [0]:
from pyspark.sql.functions import col

df = df.withColumn("revenue", col("price") * col("quantity"))

In [0]:
from pyspark.sql.functions import to_date, sum as _sum

daily_revenue = (
    df.withColumn("date", to_date("timestamp"))
      .groupBy("date")
      .agg(_sum("revenue").alias("total_revenue"))
)

In [0]:
store_revenue = (
    df.groupBy("store_id")
      .agg(_sum("revenue").alias("total_revenue"))
)

In [0]:
product_performance = (
    df.groupBy("product_id")
      .agg(
          _sum("revenue").alias("total_revenue"),
          _sum("quantity").alias("total_units")
      )
      .orderBy(col("total_revenue").desc())
)

In [0]:
customer_spend = (
    df.groupBy("customer_id")
      .agg(_sum("revenue").alias("lifetime_value"))
      .orderBy(col("lifetime_value").desc())
)

In [0]:
payment_breakdown = (
    df.groupBy("payment_method")
      .agg(_sum("revenue").alias("total_revenue"))
)

In [0]:
top_products = product_performance.limit(10)

top_products.write.format("delta").mode("overwrite").saveAsTable("gold_top_products")

In [0]:
daily_revenue.write.format("delta").mode("overwrite").saveAsTable("gold_daily_revenue")

store_revenue.write.format("delta").mode("overwrite").saveAsTable("gold_store_revenue")

product_performance.write.format("delta").mode("overwrite").saveAsTable("gold_product_performance")

customer_spend.write.format("delta").mode("overwrite").saveAsTable("gold_customer_ltv")

payment_breakdown.write.format("delta").mode("overwrite").saveAsTable("gold_payment_breakdown")


In [0]:
display(daily_revenue)
display(product_performance)
display(store_revenue)